In [1]:
#1. Setup

import os, re, time, random
from dotenv import load_dotenv
from pathlib import Path

import numpy as np
import pandas as pd
from tqdm.auto import tqdm
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report
from sklearn.metrics.pairwise import cosine_similarity
from cerebras.cloud.sdk import Cerebras

# Load environment variables
load_dotenv()

# Configs
ROLE_MAP = {"A": "Abuser", "V": "Victim", "T": "Third"}
ROLE_LABELS = ["Abuser", "Victim", "Third"]
SEED = 42
TEST_SIZE = 200
MODEL_NAME  = "llama3.1-8b" 
REQUEST_DELAY_SECONDS = 0.3


# Cerebras client
client = Cerebras(
    api_key=os.environ.get("CEREBRAS_API_KEY")
)

# preprocessing
def resolve_data_path() -> Path:
    candidates = [
        Path.cwd() / "reviewer.csv",
        Path.cwd() / "HWK2" / "reviewer.csv",
        Path.cwd().parent / "HWK2" / "reviewer.csv",
    ]
    for p in candidates:
        if p.exists():
            return p
    raise FileNotFoundError("can't find reviewer.csv, please check the path.")

DATA_PATH = resolve_data_path()
print("Data path :", DATA_PATH)
print("Model     :", MODEL_NAME)
print("API key   : OK" if os.getenv("CEREBRAS_API_KEY") else "API key   : MISSING")
print("API key prefix:", os.environ.get("CEREBRAS_API_KEY")[:6])

c:\Users\alvin\OneDrive\Documents\VS Code Projects\Natural Language Processing\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Data path : c:\Users\alvin\OneDrive\Documents\VS Code Projects\Natural Language Processing\HWK2\reviewer.csv
Model     : llama3.1-8b
API key   : OK
API key prefix: csk-3y


In [2]:
#2. load reviewer.csv, map roles, stratified split

def clean_text(text: str) -> str:
    return re.sub(r"\s+", " ", str(text)).strip()

df_raw = pd.read_csv(DATA_PATH)

role_col = "Victim/Abuser/Third"
text_col = "body"

df = df_raw[[text_col, role_col]].dropna().copy()
df[role_col] = df[role_col].map(ROLE_MAP).fillna(df[role_col])
df["text"] = df[text_col].apply(clean_text)
df = df[df[role_col].isin(ROLE_LABELS)].reset_index(drop=True)

print(f"Total labeled rows: {len(df)}")

# Stratified split
source_df, test_df = train_test_split(
    df,
    test_size=200,
    stratify=df[role_col],
    random_state=SEED,
)


source_df = source_df.reset_index(drop=True)
test_df   = test_df.reset_index(drop=True)

print(f"Source set : {len(source_df)} rows")
print(f"Test set   : {len(test_df)} rows\n")

print("Source role distribution:")
print(source_df[role_col].value_counts(), "\n")
print("Test role distribution:")
print(test_df[role_col].value_counts())

Total labeled rows: 403
Source set : 203 rows
Test set   : 200 rows

Source role distribution:
Victim/Abuser/Third
Abuser    112
Victim     61
Third      30
Name: count, dtype: int64 

Test role distribution:
Victim/Abuser/Third
Abuser    111
Victim     60
Third      29
Name: count, dtype: int64


In [3]:
#3. helper: call Cerebras chat API

def call_cerebras_chat(prompt: str, max_tokens: int = 64) -> str:

    resp = client.chat.completions.create(
        model=MODEL_NAME,
        messages=[
            {"role": "user", "content": prompt},
        ],
        max_tokens=max_tokens,
        temperature=0.0,
    )
    return resp.choices[0].message.content.strip()

In [4]:
#4. zero-shot role classifier

def classify_role_zero_shot(text: str) -> str:
    prompt = f"""
You are a classifier that reads app reviews about surveillance or tracking apps.
Your task is to decide the narrator's role:

- Abuser: someone using the app to control, monitor, or stalk others.
- Victim: someone being monitored, stalked, or harmed by others.
- Third-party: a neutral reviewer, observer, or bystander commenting on such apps.

Review:
{text}

Answer with exactly one word: Abuser, Victim, or Third.
"""

    raw = call_cerebras_chat(prompt)
    ans = raw.strip()

    # simple normalize：check if the answer contains any of the three labels
    for r in ROLE_LABELS:
        if r.lower() in ans.lower():
            return r
    return "Unknown"

In [5]:
#5. run zero-shot on all 200 test samples

zero_preds = []
for txt in tqdm(test_df["text"], desc="Zero-shot on test"):
    pred = classify_role_zero_shot(txt)
    zero_preds.append(pred)
    time.sleep(0.1)

test_df["pred_zero_shot"] = zero_preds

print(test_df[["text", "Victim/Abuser/Third", "pred_zero_shot"]].head())

Zero-shot on test: 100%|██████████| 200/200 [05:54<00:00,  1.77s/it]

                                                text Victim/Abuser/Third  \
0  I would like to have stealth mode where your n...               Third   
1   Great app to stalk your teenagers with! Love it!              Abuser   
2  Love that your girlfriend or other crazy chick...              Victim   
3  As a teenager we should have freedom most pare...              Victim   
4    BEST APP EVER I CAN SPY ON EVERYBODY AND PRANKS              Abuser   

  pred_zero_shot  
0         Abuser  
1         Abuser  
2         Abuser  
3         Victim  
4         Abuser  


In [6]:
from sklearn.metrics import classification_report

print(classification_report(
    test_df["Victim/Abuser/Third"],
    test_df["pred_zero_shot"],
    labels=ROLE_LABELS,
    zero_division=0,
))

              precision    recall  f1-score   support

      Abuser       0.73      0.86      0.79       111
      Victim       0.76      0.63      0.69        60
       Third       0.25      0.17      0.20        29

    accuracy                           0.69       200
   macro avg       0.58      0.55      0.56       200
weighted avg       0.67      0.69      0.67       200



In [7]:
# Save zero-shot predictions to CSV

# Only save if pred_zero_shot exists
if "pred_zero_shot" in test_df.columns:
    out_df = test_df[["text", "Victim/Abuser/Third", "pred_zero_shot"]].copy()
    out_df.columns = ["text", "role", "pred_zero_shot"]
    out_df["idx"] = range(len(out_df))
    out_df.to_csv("zero_shot_predictions.csv", index=False)
    print(f"Saved {len(out_df)} predictions to zero_shot_predictions.csv")
else:
    print("Warning: pred_zero_shot column not found. Run cell #5 first to generate predictions.")

Saved 200 predictions to zero_shot_predictions.csv
